# NB_patient_360
Builds `<catalog>.gold.patient_360` from the silver layer per STTM `sttm_silver_to_gold.xlsx` (sheet `patient_360`).

* **Grain:** one row per patient
* **Driving table:** `silver.patients`
* **Joins (LEFT):** `silver.hospitals` (registered_hospital_id = hospital_id), `silver.insurance_providers` (insurance_provider_id), aggregated `silver.lab_results` (by patient_id)
* **Filters:** none
* **Load type:** FULL — `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
import re

dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip()
if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: {catalog!r}")

src = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"
print("Target:", target_table)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH lab_agg AS (
    SELECT
        patient_id,
        COUNT(*)                              AS total_tests,
        COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')  AS critical_test_count,
        MAX(test_date)                        AS last_test_date
    FROM {src}.lab_results
    GROUP BY patient_id
)
SELECT
    CAST(p.patient_id AS INT)                                                     AS patient_id,
    CAST(p.first_name AS STRING)                                                  AS first_name,
    CAST(p.last_name AS STRING)                                                   AS last_name,
    CAST(p.date_of_birth AS DATE)                                                 AS date_of_birth,
    CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
    CAST(p.gender AS STRING)                                                      AS gender,
    CAST(p.city AS STRING)                                                        AS city,
    CAST(p.state AS STRING)                                                       AS state,
    CAST(p.registered_hospital_id AS INT)                                         AS hospital_id,
    CAST(h.hospital_name AS STRING)                                               AS hospital_name,
    CAST(p.insurance_provider_id AS INT)                                          AS insurance_provider_id,
    CAST(i.provider_name AS STRING)                                               AS insurance_provider_name,
    CAST(COALESCE(l.total_tests, 0) AS INT)                                       AS total_tests,
    CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                               AS abnormal_test_count,
    CAST(COALESCE(l.critical_test_count, 0) AS INT)                               AS critical_test_count,
    CAST(l.last_test_date AS DATE)                                                AS last_test_date
FROM {src}.patients p
LEFT JOIN {src}.hospitals h
       ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {src}.insurance_providers i
       ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
       ON p.patient_id = l.patient_id
""")

In [0]:
# src_cnt = spark.table(f"{src}.patients").count()
# tgt = spark.table(target_table)
# tgt_cnt = tgt.count()
# dup_cnt = tgt_cnt - tgt.select("patient_id").distinct().count()

# print(f"silver.patients rows: {src_cnt} | gold.patient_360 rows: {tgt_cnt} | duplicate patient_id: {dup_cnt}")
# assert tgt_cnt == src_cnt, "Row count mismatch: patients must never be dropped"
# assert dup_cnt == 0, "Grain violated: duplicate patient_id"

# display(tgt.limit(10))

In [0]:
updated_df = spark.table(target_table)
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))